# Week 3 — Ask for a qualitative suggestion in JSON

**Research task:** Ask for a provisional theme, a source excerpt ID and a question for the researcher, then return to the cited passage.

**Python introduced:** a route selector, `if/else`, lists of dictionaries, `json.dumps(...)` and `json.loads(...)`.

Work through input → messages → route → call → raw return → parsed output → check. Predict each output before running its cell. The assessed routine below is the same routine printed in the coursebook and task file.

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/cjbarrie/GenAI_Soc2026/blob/main/workbook/session03/session03_qualitative_interpretation.ipynb)

Run the setup cell immediately below before doing anything else. Colab installs the required Python packages and uses OpenRouter. Ollama is used from local JupyterLab or VS Code because Colab cannot reach the Ollama server on your computer.


## Prepare the notebook environment

Run the next cell before any other code. In Colab it installs the Python SDKs used by this notebook, downloads the public course repository and selects OpenRouter because Colab cannot reach the Ollama server on your computer. On a local machine it installs nothing silently: it checks that the notebook is using the course environment and gives the exact repair command if it is not.

The setup also adds the repository root to Python's import path. This is necessary for Week 4's supplied image utility and prevents a second kind of `ModuleNotFoundError` after the repository has been cloned.


In [ ]:
SESSION = "session03"

# Run this cell first. It prepares Colab or checks the local Python environment.
import importlib as setup_importlib
import os as setup_os
import subprocess as setup_subprocess
import sys as setup_sys
from pathlib import Path as SetupPath

try:
    import google.colab as setup_colab  # type: ignore[import-not-found]
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

course_packages = {
    "openrouter": "openrouter>=0.6,<1",
    "ollama": "ollama>=0.6,<1",
}
if SESSION == "session13":
    course_packages["pandas"] = "pandas>=2.2,<3"

missing_packages = [
    package_name
    for package_name in course_packages
    if setup_importlib.util.find_spec(package_name) is None
]

if IN_COLAB:
    if missing_packages:
        packages_to_install = [course_packages[name] for name in missing_packages]
        setup_subprocess.run(
            [
                setup_sys.executable,
                "-m",
                "pip",
                "install",
                "--quiet",
                "--disable-pip-version-check",
                *packages_to_install,
            ],
            check=True,
        )
        setup_importlib.invalidate_caches()

    COURSE_ROOT = SetupPath(
        setup_os.getenv("COURSE_COLAB_ROOT", "/content/GenAI_Soc2026")
    )
    if not COURSE_ROOT.exists():
        setup_subprocess.run(
            [
                "git",
                "clone",
                "--depth",
                "1",
                "https://github.com/cjbarrie/GenAI_Soc2026.git",
                str(COURSE_ROOT),
            ],
            check=True,
        )
    setup_os.chdir(COURSE_ROOT / "workbook" / SESSION)
else:
    COURSE_ROOT = SetupPath.cwd()
    while not (COURSE_ROOT / "config" / "course_models.json").exists() and COURSE_ROOT != COURSE_ROOT.parent:
        COURSE_ROOT = COURSE_ROOT.parent
    if missing_packages:
        missing_text = ", ".join(missing_packages)
        raise ModuleNotFoundError(
            f"This notebook is using a Python environment without: {missing_text}.\n\n"
            "Close Jupyter. Open a terminal in the GenAI_Soc2026 repository and run:\n"
            "    uv sync\n"
            "    uv run jupyter lab\n\n"
            "In VS Code, select the Python interpreter inside the repository's .venv folder."
        )
    if not (COURSE_ROOT / "config" / "course_models.json").exists():
        raise FileNotFoundError(
            "The course repository root could not be found. Start Jupyter from the "
            "GenAI_Soc2026 folder with: uv run jupyter lab"
        )

course_root_text = str(COURSE_ROOT)
if course_root_text not in setup_sys.path:
    setup_sys.path.insert(0, course_root_text)

still_missing = [
    package_name
    for package_name in course_packages
    if setup_importlib.util.find_spec(package_name) is None
]
if still_missing:
    raise ModuleNotFoundError(
        "Setup did not make these packages available: " + ", ".join(still_missing)
    )

print("Environment:", "Google Colab" if IN_COLAB else "local course environment")
print("Course root:", COURSE_ROOT)
print("Working folder:", SetupPath.cwd())
print("Python SDKs: ready")
if IN_COLAB:
    print("Route for this runtime: OpenRouter")
    print("Ollama work: complete later in local JupyterLab or on the in-class machine")


In [ ]:
import json
import os
from getpass import getpass
from pathlib import Path

import ollama
from openrouter import OpenRouter

ROOT = Path.cwd()
while not (ROOT / "config" / "course_models.json").exists() and ROOT != ROOT.parent:
    ROOT = ROOT.parent

config = json.loads((ROOT / "config" / "course_models.json").read_text())
HOSTED_MODEL = config["hosted"]["model"]
LOCAL_MODEL = config["local"]["model"]

if not os.getenv("OPENROUTER_API_KEY"):
    os.environ["OPENROUTER_API_KEY"] = getpass("OpenRouter course key (hidden): ")

print("Hosted model:", HOSTED_MODEL)
print("Local model:", LOCAL_MODEL)

## Choose one route and store two identified excerpts

`ROUTE` is a string controlling which later branch runs. `excerpts` is a list containing two dictionaries; every dictionary keeps an excerpt ID beside its text. Keeping IDs with passages makes it possible to test whether a model's cited evidence actually exists.

In Colab, `ROUTE` is set to `"openrouter"` automatically. On a local machine it defaults to `"ollama"`; you may change it to OpenRouter.

In Colab, `ROUTE` is set to `"openrouter"` automatically. On a local machine it defaults to `"ollama"`; you may change it to OpenRouter.

In Colab, `ROUTE` is set to `"openrouter"` automatically. On a local machine it defaults to `"ollama"`; you may change it to OpenRouter.


In [ ]:
ROUTE = "openrouter" if IN_COLAB else "ollama"  # Ollama is the local default
excerpts = [
    {"id": "e01", "text": "After several exchanges, I attended the tenants' meeting."},
    {"id": "e02", "text": "I accepted help but did not attend political meetings."},
]
print("Route:", ROUTE)
print("First excerpt:", excerpts[0])


## Convert the excerpts into prompt text and construct messages

`json.dumps(excerpts)` converts the list of source dictionaries to text without discarding their IDs. The prompt states the three required output fields. The result is placed in the familiar one-message list. Its output is model input, not a theme or finding.


In [ ]:
prompt = (
    "Suggest one provisional theme. Return JSON with exactly theme, evidence_id, "
    "and question_for_researcher. Excerpts: " + json.dumps(excerpts)
)
messages = [{"role": "user", "content": prompt}]
print(prompt)

## Make the selected route's call without hiding either branch

`if ROUTE == "openrouter"` runs only when that comparison is true; otherwise `else` runs the Ollama code. Both complete calls remain visible. Each branch assigns returned text to the same name, `raw_json`, so the parsing step below is identical whichever route was chosen.


In [ ]:
if ROUTE == "openrouter":
    with OpenRouter(api_key=os.environ["OPENROUTER_API_KEY"]) as client:
        response = client.chat.send(
            model=HOSTED_MODEL,
            messages=messages,
            temperature=0,
            response_format={"type": "json_object"},
        )
    raw_output = response.choices[0].message.content
else:
    response = ollama.chat(
        model=LOCAL_MODEL,
        messages=messages,
        format="json",
        options={"temperature": 0},
    )
    raw_output = response.message.content

print("Raw JSON text:", raw_output)

## Parse the JSON string and return to its cited source

`json.loads(raw_json)` parses the returned string into a Python dictionary. Square brackets retrieve the theme and cited ID. The `for` loop checks each source record until its ID matches; it then saves the actual excerpt. The output to assess is the pair `theme` plus `cited_excerpt`, not the theme alone.


In [ ]:
suggestion = json.loads(raw_output)
evidence_id = suggestion["evidence_id"]
cited_excerpt = None
for excerpt in excerpts:
    if excerpt["id"] == evidence_id:
        cited_excerpt = excerpt

print("Theme:", suggestion["theme"])
print("Cited excerpt:", cited_excerpt)
print("Question:", suggestion["question_for_researcher"])

# ONE CHANGE: change e02 to
# "The food deliveries helped, but I avoided the group because meetings felt hostile."

## Methodological check

Well-formed JSON makes the fields retrievable. The researcher must still decide whether the cited excerpt supports the theme and what contrary evidence changes it.
## Completion recording

Use one chosen route, change only e02 and rerun. Explain `ROUTE`, both `if/else` branches, `json.dumps`, the raw returned string, `json.loads` and the source lookup. Say whether the cited passage supports the proposed theme.

Explain every input and output aloud. Never show the shared key.